In [1]:
"""
make_sample.py
ينشئ عينة التدريب (600 صورة: 200 لكل دولة) من Dataset roadscan-yolo.
يشتغل داخل دفتر Kaggle بعد ربط الـ Dataset.
seed ثابت = 42، عشان تطلع نفس العينة عند الجميع.
"""
import os, random, shutil

SRC = "/kaggle/input/datasets/sarvii7/roadscan-yolo/RoadScan_yolo"
DST = "/kaggle/working/sample600"
SEED = 42
PER_COUNTRY = 200
COUNTRIES = ["Japan", "India", "Czech"]
MAX_EMPTY_FRAC = 0.15   # أقصى نسبة صور بدون ضرر

random.seed(SEED)
for part in ["train", "val"]:
    os.makedirs(f"{DST}/{part}/images", exist_ok=True)
    os.makedirs(f"{DST}/{part}/labels", exist_ok=True)

img_dir = f"{SRC}/train/images"
lbl_dir = f"{SRC}/train/labels"
names = sorted(os.listdir(img_dir))

chosen = []
for c in COUNTRIES:
    pool = [n[:-4] for n in names if n.startswith(c + "_") and n.endswith(".jpg")]
    full, empty = [], []
    for stem in pool:
        size = os.path.getsize(f"{lbl_dir}/{stem}.txt")
        (full if size > 0 else empty).append(stem)
    n_empty = int(PER_COUNTRY * MAX_EMPTY_FRAC)
    pick = random.sample(full, PER_COUNTRY - n_empty) + random.sample(empty, n_empty)
    chosen += pick
    print(c, "| with damage:", PER_COUNTRY - n_empty, "| empty:", n_empty)

for stem in chosen:
    shutil.copy(f"{img_dir}/{stem}.jpg", f"{DST}/train/images/{stem}.jpg")
    shutil.copy(f"{lbl_dir}/{stem}.txt", f"{DST}/train/labels/{stem}.txt")

# val (الصين) ننسخه كاملاً
for n in os.listdir(f"{SRC}/val/images"):
    shutil.copy(f"{SRC}/val/images/{n}", f"{DST}/val/images/{n}")
    shutil.copy(f"{SRC}/val/labels/{n[:-4]}.txt", f"{DST}/val/labels/{n[:-4]}.txt")

with open("/kaggle/working/sample600_list.txt", "w") as f:
    f.write("\n".join(sorted(chosen)))

yaml = f"""path: {DST}
train: train/images
val: val/images
names:
  0: D00
  1: D10
  2: D20
  3: D40
"""
open("/kaggle/working/data.yaml", "w").write(yaml)
print("train:", len(chosen), "| val:", len(os.listdir(f"{DST}/val/images")))

Japan | with damage: 170 | empty: 30
India | with damage: 170 | empty: 30
Czech | with damage: 170 | empty: 30
train: 600 | val: 1977


In [2]:
import os
from collections import Counter

root = "/kaggle/working/sample600"
names = ["D00", "D10", "D20", "D40"]

for split in ["train", "val"]:
    lbl_dir = f"{root}/{split}/labels"
    files = os.listdir(lbl_dir)
    img_count = Counter()
    empty = 0
    for f in files:
        with open(f"{lbl_dir}/{f}") as fh:
            lines = [l.split() for l in fh.read().strip().splitlines() if l.strip()]
        if not lines:
            empty += 1
        for c in {int(l[0]) for l in lines}:
            img_count[c] += 1
    print(f"{split} | صور: {len(files)} | فاضية: {empty}")
    for c, name in enumerate(names):
        print(f"   {name}: {img_count[c]} صورة")

train | صور: 600 | فاضية: 90
   D00: 244 صورة
   D10: 102 صورة
   D20: 209 صورة
   D40: 125 صورة
val | صور: 1977 | فاضية: 43
   D00: 1419 صورة
   D10: 738 صورة
   D20: 486 صورة
   D40: 164 صورة


In [3]:
import os, numpy as np

root = "/kaggle/working/sample600"

def build_vectors(split):
    lbl_dir = f"{root}/{split}/labels"
    stems, vecs = [], []
    for f in sorted(os.listdir(lbl_dir)):
        v = np.zeros(4, dtype=np.float32)
        with open(f"{lbl_dir}/{f}") as fh:
            for line in fh.read().strip().splitlines():
                if line.strip():
                    v[int(line.split()[0])] = 1
        stems.append(f[:-4])
        vecs.append(v)
    return stems, np.stack(vecs)

train_stems, train_y = build_vectors("train")
val_stems, val_y = build_vectors("val")

print(train_y.shape, val_y.shape)
print(train_y[:5])
print(train_y.sum(axis=0), val_y.sum(axis=0))

(600, 4) (1977, 4)
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
[244. 102. 209. 125.] [1419.  738.  486.  164.]


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

tf = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class RoadDataset(Dataset):
    def __init__(self, split, stems, y):
        self.split, self.stems, self.y = split, stems, y
    def __len__(self):
        return len(self.stems)
    def __getitem__(self, i):
        img = Image.open(f"{root}/{self.split}/images/{self.stems[i]}.jpg").convert("RGB")
        return tf(img), torch.tensor(self.y[i])

train_ds = RoadDataset("train", train_stems, train_y)
val_ds = RoadDataset("val", val_stems, val_y)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

x, y = next(iter(train_dl))
print(x.shape, y.shape)

torch.Size([32, 3, 384, 384]) torch.Size([32, 4])


In [5]:
import torch
print(torch.cuda.is_available())

True


In [6]:
print(len(train_ds), len(val_ds))

600 1977


In [7]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 4)

# المرحلة 1: نجمّد كل الطبقات ما عدا الأخيرة
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("trainable:", trainable, "| total:", total)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 250MB/s]


trainable: 2052 | total: 11178564


In [8]:
import torch.optim as optim

torch.manual_seed(42)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

def run_epoch(dl, train=True):
    model.train(train)
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            n += x.size(0)
    return total_loss / n

for epoch in range(1, 6):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    print(f"epoch {epoch} | train loss {tr:.4f} | val loss {va:.4f}")

epoch 1 | train loss 0.5889 | val loss 0.7125
epoch 2 | train loss 0.5549 | val loss 0.7587
epoch 3 | train loss 0.5346 | val loss 0.8216
epoch 4 | train loss 0.5222 | val loss 0.8993
epoch 5 | train loss 0.5152 | val loss 0.8878


In [9]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

@torch.no_grad()
def predict(dl):
    model.eval()
    probs = []
    for x, _ in dl:
        probs.append(torch.sigmoid(model(x.to(device))).cpu().numpy())
    return np.concatenate(probs)

val_probs = predict(val_dl)
pred = (val_probs >= 0.5).astype(int)

p, r, f1, _ = precision_recall_fscore_support(val_y, pred, zero_division=0)
for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(f"{name} | precision {p[i]:.2f} | recall {r[i]:.2f} | F1 {f1[i]:.2f}")

D00 | precision 0.00 | recall 0.00 | F1 0.00
D10 | precision 0.00 | recall 0.00 | F1 0.00
D20 | precision 0.25 | recall 0.98 | F1 0.40
D40 | precision 0.00 | recall 0.00 | F1 0.00


In [10]:
from sklearn.metrics import roc_auc_score

print("متوسط الاحتمال لكل فئة:", val_probs.mean(axis=0).round(3))
print("أعلى احتمال لكل فئة:  ", val_probs.max(axis=0).round(3))
for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(name, "AUC:", round(roc_auc_score(val_y[:, i], val_probs[:, i]), 3))

متوسط الاحتمال لكل فئة: [0.196 0.122 0.736 0.113]
أعلى احتمال لكل فئة:   [0.413 0.279 0.903 0.433]
D00 AUC: 0.449
D10 AUC: 0.627
D20 AUC: 0.452
D40 AUC: 0.502


In [11]:
train_eval_dl = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2)
train_probs = predict(train_eval_dl)
for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(name, "train AUC:", round(roc_auc_score(train_y[:, i], train_probs[:, i]), 3))

D00 train AUC: 0.68
D10 train AUC: 0.751
D20 train AUC: 0.771
D40 train AUC: 0.795


In [12]:
for p in model.parameters():
    p.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(1, 6):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    print(f"epoch {epoch} | train loss {tr:.4f} | val loss {va:.4f}")

val_probs = predict(val_dl)
for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(name, "val AUC:", round(roc_auc_score(val_y[:, i], val_probs[:, i]), 3))

epoch 1 | train loss 0.5083 | val loss 0.8521
epoch 2 | train loss 0.3392 | val loss 1.0019
epoch 3 | train loss 0.2116 | val loss 0.9936
epoch 4 | train loss 0.1177 | val loss 1.3042
epoch 5 | train loss 0.0648 | val loss 1.2923
D00 val AUC: 0.515
D10 val AUC: 0.496
D20 val AUC: 0.676
D40 val AUC: 0.615


In [13]:
import copy

tf_aug = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class RoadDatasetAug(RoadDataset):
    def __getitem__(self, i):
        img = Image.open(f"{root}/{self.split}/images/{self.stems[i]}.jpg").convert("RGB")
        return tf_aug(img), torch.tensor(self.y[i])

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 4)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.fc.parameters():
        p.requires_grad = True
    return m.to(device)

def experiment(train_dataset, epochs=8):
    global model, optimizer
    torch.manual_seed(42)
    model = build_model()
    optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
    dl = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    best_va, best_state = 1e9, None
    for epoch in range(1, epochs + 1):
        tr = run_epoch(dl, True)
        va = run_epoch(val_dl, False)
        print(f"epoch {epoch} | train {tr:.4f} | val {va:.4f}")
        if va < best_va:
            best_va, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    probs = predict(val_dl)
    aucs = [roc_auc_score(val_y[:, i], probs[:, i]) for i in range(4)]
    print("best val loss:", round(best_va, 4), "| val AUC:", np.round(aucs, 3))
    return best_va, aucs

print("=== A: بدون augmentation ===")
resA = experiment(train_ds)
print("=== B: مع augmentation ===")
resB = experiment(RoadDatasetAug("train", train_stems, train_y))

=== A: بدون augmentation ===
epoch 1 | train 0.6191 | val 0.6646
epoch 2 | train 0.5630 | val 0.6901
epoch 3 | train 0.5446 | val 0.6768
epoch 4 | train 0.5328 | val 0.7388
epoch 5 | train 0.5214 | val 0.8078
epoch 6 | train 0.5085 | val 0.7689
epoch 7 | train 0.5047 | val 0.7429
epoch 8 | train 0.4977 | val 0.8791
best val loss: 0.6646 | val AUC: [0.403 0.526 0.572 0.464]
=== B: مع augmentation ===
epoch 1 | train 0.6217 | val 0.6656
epoch 2 | train 0.5656 | val 0.6851
epoch 3 | train 0.5497 | val 0.6686
epoch 4 | train 0.5367 | val 0.7242
epoch 5 | train 0.5268 | val 0.7884
epoch 6 | train 0.5129 | val 0.7567
epoch 7 | train 0.5107 | val 0.7255
epoch 8 | train 0.5044 | val 0.8615
best val loss: 0.6656 | val AUC: [0.398 0.529 0.581 0.473]


In [14]:
from sklearn.model_selection import train_test_split

# نقسم بحسب الدولة (اسم الصورة يبدأ باسم الدولة) عشان التوزيع يبقى متوازن
countries = [s.split("_")[0] for s in train_stems]
idx = np.arange(len(train_stems))
tr_idx, ho_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=countries)

sub_train = RoadDataset("train", [train_stems[i] for i in tr_idx], train_y[tr_idx])
holdout   = RoadDataset("train", [train_stems[i] for i in ho_idx], train_y[ho_idx])
holdout_dl = DataLoader(holdout, batch_size=64, shuffle=False, num_workers=2)
holdout_y = train_y[ho_idx]

torch.manual_seed(42)
model = build_model()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
dl = DataLoader(sub_train, batch_size=32, shuffle=True, num_workers=2)

for epoch in range(1, 9):
    tr = run_epoch(dl, True)
    ho = run_epoch(holdout_dl, False)
    print(f"epoch {epoch} | train {tr:.4f} | holdout {ho:.4f}")

ho_probs = predict(holdout_dl)
vl_probs = predict(val_dl)
for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(name,
          "| AUC holdout (نفس الدول):", round(roc_auc_score(holdout_y[:, i], ho_probs[:, i]), 3),
          "| AUC val (الصين):", round(roc_auc_score(val_y[:, i], vl_probs[:, i]), 3))

epoch 1 | train 0.6404 | holdout 0.5736
epoch 2 | train 0.5761 | holdout 0.5425
epoch 3 | train 0.5538 | holdout 0.5333
epoch 4 | train 0.5413 | holdout 0.5121
epoch 5 | train 0.5287 | holdout 0.5123
epoch 6 | train 0.5192 | holdout 0.5114
epoch 7 | train 0.5129 | holdout 0.4943
epoch 8 | train 0.5085 | holdout 0.5040
D00 | AUC holdout (نفس الدول): 0.598 | AUC val (الصين): 0.375
D10 | AUC holdout (نفس الدول): 0.691 | AUC val (الصين): 0.68
D20 | AUC holdout (نفس الدول): 0.767 | AUC val (الصين): 0.549
D40 | AUC holdout (نفس الدول): 0.864 | AUC val (الصين): 0.513


In [15]:
import copy
from sklearn.metrics import precision_recall_fscore_support

# 1) نعيد تدريب النموذج على الـ600 كاملة، ونحفظ أفضل epoch حسب val loss
torch.manual_seed(42)
model = build_model()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
best_va, best_state = 1e9, None
for epoch in range(1, 9):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    if va < best_va:
        best_va, best_state = va, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print("best val loss:", round(best_va, 4))

# 2) نضبط الحد لكل فئة على val
val_probs = predict(val_dl)
names = ["D00", "D10", "D20", "D40"]
grid = np.arange(0.05, 0.96, 0.05)
best_th = []
for i in range(4):
    f1s = [precision_recall_fscore_support(val_y[:, i], (val_probs[:, i] >= t).astype(int),
                                           average="binary", zero_division=0)[2] for t in grid]
    best_th.append(float(grid[int(np.argmax(f1s))]))
print("الحدود المختارة:", dict(zip(names, np.round(best_th, 2))))

# 3) النتائج النهائية بالحدود المضبوطة
pred = np.stack([(val_probs[:, i] >= best_th[i]).astype(int) for i in range(4)], axis=1)
p, r, f1, _ = precision_recall_fscore_support(val_y, pred, zero_division=0)
for i, n in enumerate(names):
    print(f"{n} | th {best_th[i]:.2f} | precision {p[i]:.2f} | recall {r[i]:.2f} | F1 {f1[i]:.2f}")
print("macro-F1:", round(f1.mean(), 3))

# 4) مرجع: نسبة الفئة في val (F1 لمن يقول "موجودة" دايماً)
prev = val_y.mean(axis=0)
print("F1 لو قلنا موجودة دايماً:", np.round(2*prev/(1+prev), 2))

best val loss: 0.6646
الحدود المختارة: {'D00': np.float64(0.05), 'D10': np.float64(0.05), 'D20': np.float64(0.05), 'D40': np.float64(0.1)}
D00 | th 0.05 | precision 0.72 | recall 1.00 | F1 0.84
D10 | th 0.05 | precision 0.38 | recall 1.00 | F1 0.55
D20 | th 0.05 | precision 0.25 | recall 1.00 | F1 0.39
D40 | th 0.10 | precision 0.08 | recall 1.00 | F1 0.15
macro-F1: 0.482
F1 لو قلنا موجودة دايماً: [0.84 0.54 0.39 0.15]


In [16]:
import json
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
torch.save(model.state_dict(), "/kaggle/working/artifacts/baseline_resnet18.pt")
json.dump({"thresholds": dict(zip(names, best_th)), "seed": 42, "img_size": 384,
           "mean": mean, "std": std, "classes": names},
          open("/kaggle/working/artifacts/baseline_config.json", "w"), indent=2)
print(os.listdir("/kaggle/working/artifacts"))

['baseline_resnet18.pt', 'baseline_config.json']
